In [1]:
import numpy as np
import tifffile
import skimage as ski
import pandas as pd
import os
import os.path as osp
import matplotlib.pyplot as plt

from glob import glob
from tifffile import imread, imwrite
import tifffile
import matplotlib.pyplot as plt
from skimage import io, morphology, segmentation, measure, feature, filters, color
from scipy import ndimage as ndi
import numpy as np
from glob import glob
from cellpose import models, core, io, plot
from tqdm.notebook import trange, tqdm
import concurrent
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
import gc
import torch

import numpy as np
import matplotlib.pyplot as plt
from skimage import filters, morphology, feature, segmentation, measure
from scipy import ndimage as ndi

# %matplotlib inline



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	linux 
python version: 	3.12.12 
torch version:  	2.9.1+cu130! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




In [2]:
# data_root = "/mnt/aquila/ssd_processing/Data2601-/MOSAIC_Data/Processed_Data/20260129 Andre 3 color Cell Death/Sample 1"
infile = "/mnt/aquila/ssd_processing/Data2601-/MOSAIC_Data/Processed_Data/20260129 Andre 3 color Cell Death/Sample 1/1/MIP/MIP_processed_combined.tif"

proc_data_root = "/mnt/aquila/ssd_processing/Data2601-/MOSAIC_Data/Processed_Data/"
data_dir = "20260129 Andre 3 color Cell Death"

nuc_ch = 2
mito_ch = 0

In [3]:
#===== Helper Functions =====
def normalize_local_contrast(img,
                             sigma=4, # Gaussian σ (in pixels) that defines “local”
                             epsilon=1e-8,      # Prevents divide-by-zero
                             stretch=(0, 1)):   # Output range; None -> keep float

    # --- 1. local mean -------------------------------------------------------
    mu = ndi.gaussian_filter(img, sigma=sigma, mode='reflect')

    # --- 2. local variance & std --------------------------------------------
    mu_sq = ndi.gaussian_filter(img**2, sigma=sigma, mode='reflect')
    sigma_local = np.sqrt(np.clip(mu_sq - mu**2, 0, None)) + epsilon

    # --- 3. normalise --------------------------------------------------------
    lcn = (img - mu) / sigma_local

    # optional: rescale to a convenient range (e.g. 0-1 for uint8 export)
    if stretch is not None:
        lo, hi = stretch
        lcn = np.clip(lcn, np.percentile(lcn, 0.01), np.percentile(lcn, 99.99))
        lcn = (lcn - lcn.min()) / (lcn.max() - lcn.min() + epsilon)
        lcn = lcn * (hi - lo) + lo

    return lcn.astype('float32')

def crop_patch(vol, center, dz=30, dy=128, dx=128):

    int_low = 0.
    int_high = float(np.percentile(vol, 99.99))
    if int_high <= int_low:  # guard against flat patches
        int_high = int_low + 1.0
    vol_cropped = np.clip(vol, int_low, int_high)
    vol_norm = (vol_cropped - int_low) / (int_high - int_low)
    vol_norm = np.clip(vol_norm, 0.0, 1.0)

    zc, yc, xc = center
    if zc < 30:
        z0, z1 = 0, 60
    elif zc > 90:
        z0, z1 = 60, 120
    else:
        z0, z1 = zc - dz, zc + dz

    y0, y1 = yc - dy, yc + dy
    x0, x1 = xc - dx, xc + dx

    # assume centers are valid; add bounds checks if needed
    return vol_norm[z0:z1, y0:y1, x0:x1]

def process_one_cell(center_xy, staygold_vol):

    patch = staygold_vol[:, center_xy[0]-128:center_xy[0]+128, center_xy[1]-128:center_xy[1]+128]
    z_indices = np.arange(patch.shape[0], dtype=float)[:, None, None]
    weighted_sum_z = np.sum(patch * z_indices)
    total_intensity = np.sum(patch)

    if total_intensity > 0:
        centroid_z = int(round(weighted_sum_z / total_intensity))
    else:
        centroid_z = patch.shape[0] // 2

    sg_cropped = crop_patch(staygold_vol, [centroid_z, center_xy[0], center_xy[1]]).astype('float32', copy=False)

    # --- StayGold ---
    int_low = 2000.0
    int_high = float(np.percentile(sg_cropped, 99.99))
    if int_high <= int_low:  # guard against flat patches
        int_high = int_low + 1.0
    sg_cropped = np.clip(sg_cropped, int_low, int_high)
    sg_norm = (sg_cropped - int_low) / (int_high - int_low)
    sg_norm = np.clip(sg_norm, 0.0, 1.0)

    # local contrast normalization
    sg_lcn = normalize_local_contrast(sg_norm, sigma=(0,32,32))

    return sg_lcn, [centroid_z, center_xy[0], center_xy[1]]

def save_movie(path, movie):
    if isinstance(movie, np.ndarray) and movie.dtype == np.float32:
        np.save(path, movie)
    else:
        np.save(path, np.asarray(movie, dtype=np.float32))

import numpy as np
import matplotlib.pyplot as plt
from skimage import filters, morphology, feature, segmentation, measure
from scipy import ndimage as ndi

def patch_centers_from_nucleus(nuc_mip,
                               show_centroids=False,
                               min_coord_dist=25,  # REFACTOR: Lowered from 60 to 25 to detect close objects
                               local_threshold_block_size=101,
                               local_threshold_sigma=80,
                               gaussian_filter_sigma=3):
    patch_centers_2d = []

    # Normalization
    nuc_norm = (nuc_mip - np.percentile(nuc_mip, 1)) / (np.percentile(nuc_mip, 99) - np.percentile(nuc_mip, 1))
    blurred = filters.gaussian(nuc_norm, sigma=gaussian_filter_sigma, preserve_range=True)

    # Thresholding
    thresh = filters.threshold_local(blurred,
                                     block_size=local_threshold_block_size,
                                     method='gaussian',
                                     param=local_threshold_sigma)
    mask = blurred > thresh

    # Clean up (Morphology)
    img_clean = morphology.remove_small_objects(mask, min_size=2500)
    img_clean = morphology.remove_small_holes(img_clean, area_threshold=5000)

    # Watershed Preparation: Distance Transform
    # This creates 'peaks' in the center of objects. Touching objects will have two distinct peaks.
    dist = ndi.distance_transform_edt(img_clean)

    #

    # REFACTOR: Removed the logic that forced 1 centroid per region.
    # We now look for local peaks in the distance map.
    # If two nuclei touch, 'dist' will have two peaks.
    # 'min_distance' ensures we don't pick up noise, but it must be small enough to separate real neighbors.
    coords = feature.peak_local_max(dist,
                                    labels=img_clean,
                                    min_distance=min_coord_dist,
                                    exclude_border=False)

    # Create markers for Watershed
    markers = np.zeros_like(dist, dtype=int)
    for i, (r, c) in enumerate(coords, start=1):
        markers[r, c] = i

    # Run Watershed
    # This expands the markers until they meet, effectively drawing a line between touching cells.
    labels = segmentation.watershed(-dist, markers, mask=img_clean)

    props = measure.regionprops(labels)
    centroids = np.array([p.centroid for p in props])

    # Filter centroids based on boundary proximity
    dim_y, dim_x = nuc_mip.shape
    if centroids.size > 0:
        for centroid in centroids:
            coord_y, coord_x = [round(centroid[0]), round(centroid[1])]
            # Boundary check
            if coord_y > 150 and coord_y < dim_y - 150 and coord_x > 300 and coord_x < dim_x - 300:
                patch_centers_2d.append([coord_y, coord_x])

    patch_centers_2d = np.array(patch_centers_2d)

    if show_centroids:
        plt.figure(figsize=(20,6))
        plt.imshow(nuc_norm, cmap='gray')
        if patch_centers_2d.size > 0:
            plt.scatter(patch_centers_2d[:, 1], patch_centers_2d[:, 0], c='red', s=15)
        plt.axis('off')
        plt.show()

    return patch_centers_2d, img_clean, labels

# def patch_centers_from_nucleus(nuc_mip,
#                                show_centroids=False,
#                                min_coord_dist=60,
#                                local_threshold_block_size=101,
#                                local_threshold_sigma=80,
#                                gaussian_filter_sigma=3):
#     patch_centers_2d = []
#
#     nuc_norm = (nuc_mip - np.percentile(nuc_mip, 1)) / (np.percentile(nuc_mip, 99) - np.percentile(nuc_mip, 1))
#     blurred = filters.gaussian(nuc_norm, sigma=gaussian_filter_sigma, preserve_range=True)
#
#     # thresholding
#     thresh = filters.threshold_local(blurred,
#                                      block_size=local_threshold_block_size, # neighborhood size
#                                      method='gaussian', # gaussian weighted mean
#                                      param=local_threshold_sigma)
#     # mask = blurred > thresh
#     img_clean = blurred > thresh
#
#     # clean up to leave nucleus-like shape
#     img_clean = morphology.remove_small_objects(img_clean, min_size=2500)
#     img_clean = morphology.remove_small_holes(img_clean, area_threshold=5000)
#
#     # watershed on distance transform to get labeled mask
#     dist = ndi.distance_transform_edt(img_clean)
#
#     coords = feature.peak_local_max(dist, labels=img_clean, min_distance=min_coord_dist)
#     markers = np.zeros_like(dist, dtype=int)
#     for i, (r, c) in enumerate(coords, start=1):
#         markers[r, c] = i
#
#     labels = segmentation.watershed(-dist, markers, mask=img_clean)
#     props = measure.regionprops(labels)
#     centroids = np.array([p.centroid for p in props])
#
#     # remove centroids too close to image boundary
#     dim_y, dim_x = nuc_mip.shape
#     for centroid in centroids:
#         coord_y, coord_x = [round(centroid[0]), round(centroid[1])]
#         if coord_y > 150 and coord_y < dim_y - 150 and coord_x > 300 and coord_x < dim_x - 300:
#             patch_centers_2d.append([coord_y, coord_x])
#
#     patch_centers_2d = np.array(patch_centers_2d)
#
#     if show_centroids:
#         plt.figure(figsize=(20,6))
#         plt.imshow(img_clean, cmap='gray')
#         plt.scatter(patch_centers_2d[:, 1], patch_centers_2d[:, 0], c='red', s=15)
#         plt.axis('off')
#         plt.show()
#
#     return patch_centers_2d, img_clean

# Testing

In [ ]:
def show_img(img, centroids=None):
    plt.figure(figsize=(20,6))
    plt.imshow(img, cmap='gray')
    if centroids:
        plt.scatter(centroids[:, 1], centroids[:, 0], c='red', s=15)
    plt.axis('off')
    plt.show()

In [ ]:
# img = tifffile.imread(infile)
# img = np.array(img)
# nuc_mip = img[-1, nuc_ch, ...]

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from skimage.feature import peak_local_max
# from skimage.segmentation import watershed
# from skimage.measure import regionprops
# from skimage.filters import threshold_local, threshold_otsu, gaussian
# from skimage.color import label2rgb
# from skimage import img_as_float
# from skimage.morphology import remove_small_objects
# from scipy import ndimage as ndi
#
# def process_and_visualize_local(image_array, block_size=101, offset=0.01, min_distance=60):
#     """
#     Identifies centroids using a combination of local and global thresholding
#     to prevent background noise detection.
#     """
#
#     # 0. Safety: Clip image to valid 0-1 range (handles outliers from your normalization)
#     image_float = img_as_float(image_array)
#     image_float = np.clip(image_float, 0, 1)
#
#     # 1. Apply Gaussian Blur
#     blurred_image = gaussian(image_float, sigma=1)
#
#     # 2. Create Masks
#     # A. Local Mask: Adapts to uneven lighting to separate close cells
#     local_thresh = threshold_local(blurred_image, block_size, offset=offset)
#     local_mask = blurred_image > local_thresh
#
#     # B. Global Mask: The "Floor" (Removes background noise)
#     # Otsu's method finds the intensity value that separates pixels into "foreground" and "background"
#     global_thresh = threshold_otsu(blurred_image)
#     global_mask = blurred_image > global_thresh
#
#     # C. Combine: A pixel must be locally bright AND globally bright enough to count
#     binary_mask = local_mask & global_mask
#
#     # 3. Cleanup
#     # Fill holes to make solid shapes
#     binary_mask = ndi.binary_fill_holes(binary_mask)
#
#     # Remove specks: filter out objects smaller than a fraction of expected cell size
#     # Adjust 'min_size' if valid small cells are being deleted
#     binary_mask = remove_small_objects(binary_mask, min_size=50)
#
#     # 4. Watershed Segmentation
#     distance_map = ndi.distance_transform_edt(binary_mask)
#
#     coords = peak_local_max(distance_map, min_distance=min_distance, labels=binary_mask)
#     mask = np.zeros(distance_map.shape, dtype=bool)
#     mask[tuple(coords.T)] = True
#     markers, _ = ndi.label(mask)
#
#     labels = watershed(-distance_map, markers, mask=binary_mask)
#
#     # 5. Calculate Centroids
#     centroids = []
#     for prop in regionprops(labels):
#         centroids.append(prop.centroid)
#
#     # --- Visualization ---
#     image_label_overlay = label2rgb(labels, image=image_array, bg_label=0)
#
#     fig, ax = plt.subplots(figsize=(15, 6))
#     ax.imshow(image_label_overlay)
#
#     rows = [c[0] for c in centroids]
#     cols = [c[1] for c in centroids]
#     ax.scatter(cols, rows, color='red', s=5, marker='.')
#
#     ax.set_title(f'Identified {len(centroids)} Regions (Local + Global Filter)')
#     ax.axis('off')
#     plt.tight_layout()
#     plt.show()
#
#     return centroids, labels
#
# # --- Run ---
# # Your normalization logic (ensure nuc_mip is defined)
# nuc_norm = (nuc_mip - np.percentile(nuc_mip, 1)) / (np.percentile(nuc_mip, 99) - np.percentile(nuc_mip, 1))
#
# centroids, labels = process_and_visualize_local(nuc_norm, block_size=101, offset=0.01, min_distance=50)

In [ ]:
# nuc_norm = (nuc_mip - np.percentile(nuc_mip, 1)) / (np.percentile(nuc_mip, 99) - np.percentile(nuc_mip, 1))
# nuc_lcn = normalize_local_contrast(nuc_norm, sigma=128)

# show_img(nuc_lcn)

# End Testing

In [ ]:
# data_path = sorted(glob(root_path + '/Data2601-/MOSAIC_Data/Processed_Data/*'+date+'*/Sample '+sample_id+'/'))[0]

In [ ]:
region_path = "/mnt/aquila/ssd_processing/Data2601-/MOSAIC_Data/Processed_Data/20260129 Andre 3 color Cell Death/Sample 1/1"

staygold_mip_list = sorted(glob(region_path+'/MIP/*488nm*.tif'))

In [ ]:
# def filter_patch_centers(staygold_list):
#     staygold_mean_list = []
#     staygold_first = imread(staygold_list[0])
#     filtered_patch_centers_2d = []
#     for cell_id, center in enumerate(patch_centers_2d):
#
#         patch = staygold_first[:, center[0]-128:center[0]+128, center[1]-128:center[1]+128]
#
#         # remove too dim/bright cells
#         staygold_mean = np.mean(patch[patch>filters.threshold_otsu(patch)])
#         staygold_mean_list.append(int(staygold_mean))
#         if staygold_mean < 20000 and staygold_mean > 5000:
#             filtered_patch_centers_2d.append(patch_centers_2d[cell_id])
#
#     ### plot selected cells ###
#     filtered_patch_centers_2d = np.array(filtered_patch_centers_2d)
#
#     cmap = plt.get_cmap('tab20')
#     colors = []
#     for i in range(labels.max()):
#         colors.append(cmap(i%20))
#
#     overlay = color.label2rgb(labels, image=staygold_lcn, colors=colors, bg_label=0, alpha=0.4)
#
#     plt.figure(figsize=(20, 6))
#     plt.imshow(overlay)
#     plt.scatter(filtered_patch_centers_2d[:, 1], filtered_patch_centers_2d[:, 0], c='green', s=15)
#     plt.axis('off')
#     plt.show()
#     print('Cell centroids determined from mitochondria')

In [4]:
root_path = '/mnt/aquila/ssd_processing/'
metadata_path = root_path+'/Others/MitoSpace4D/andre_3color_celldeath/metadata.csv'
mapfile_path = root_path+'/Others/MitoSpace4D/andre_3color_celldeath/cell_to_region.csv'

use_nucleus = True
all_date = ['20260129']
all_sample_id = ['1']
for i in range(len(all_date)):
    print(i, all_date[i], all_sample_id[i])

0 20260129 1


# Working Copy of Main Processing Loop

In [ ]:
use_nucleus = True

# Nucleus centroid params
min_coord_dist = 50 # 60
local_threshold_block_size = 101
local_threshold_sigma = 80
gaussian_filter_sigma = 3

In [ ]:
all_sample_region_patch_centers = []
for i in range(len(all_sample_id)):

    if os.path.exists(metadata_path):
        metadata = pd.read_csv(metadata_path)
    else:
        metadata = pd.DataFrame(columns=['sample id', 'region id', 'cell id', 'movie id', 'cell center (x,y)'])

    if os.path.exists(mapfile_path):
        mapfile = pd.read_csv(mapfile_path)
    else:
        mapfile = pd.DataFrame(columns=['Data Path', 'Region ID', 'Cell ID Start'])

    date = all_date[i]
    sample_id = all_sample_id[i]
    # data_path = sorted(glob(root_path + '/Data2601-/MOSAIC_Data/Processed_Data/*'+date+'*/Sample '+sample_id+'/'))[0]
    data_path = sorted(glob(root_path + 'Data2601-/MOSAIC_Data/Processed_Data/*'+date+'*Andre 3 color Cell Death/Sample '+sample_id+'/'))[0]

    print(data_path)

In [ ]:

all_sample_region_patch_centers = []
for i in range(len(all_sample_id)):

    if os.path.exists(metadata_path):
        metadata = pd.read_csv(metadata_path)
    else:
        metadata = pd.DataFrame(columns=['sample id', 'region id', 'cell id', 'movie id', 'cell center (x,y)'])

    if os.path.exists(mapfile_path):
        mapfile = pd.read_csv(mapfile_path)
    else:
        mapfile = pd.DataFrame(columns=['Data Path', 'Region ID', 'Cell ID Start'])

    date = all_date[i]
    sample_id = all_sample_id[i]
    # data_path = sorted(glob(root_path + '/Data2601-/MOSAIC_Data/Processed_Data/*'+date+'*/Sample '+sample_id+'/'))[0]
    data_path = sorted(glob(root_path + 'Data2601-/MOSAIC_Data/Processed_Data/*'+date+'*Andre 3 color Cell Death/Sample '+sample_id+'/'))[0]

    print(os.path.exists(data_path))

    all_region_paths = sorted(glob(data_path + '?'))
    # all_region_paths = [all_region_paths[0]]
    # region_ids = [int(r.split("/")[-1]) for r in all_region_paths]
    print(all_region_paths)

    offset = 0
    staygold_mean_list = []

    all_region_patch_centers = []

    for region_id, region_path in enumerate(all_region_paths):
        print('\nProcessing', region_path)

        # nuc_mip = imread(region_path+'_Nuc/MIP/MIP_processed_combined.tif')
        # mip_path = region_path+'/MIP/MIP_processed_combined.tif'
        # print(mip_path)
        # nuc_mip_movie = imread(region_path+'/MIP/MIP_processed_combined.tif')[:n_frames, nuc_ch, :, :]
        mip_movie = imread(region_path+'/MIP/MIP_processed_combined.tif')
        nuc_mip_movie = mip_movie[:, nuc_ch, :, :]
        mito_mip_movie = mip_movie[:, mito_ch, :, :]
        # nuc_mip_movie = imread(region_path+'/MIP/MIP_processed_combined.tif')[:, nuc_ch, :, :]
        staygold_3d_frame_paths = sorted(glob(region_path + '/*488nm*tif'))

        # print(f"[TESTING] Using {n_frames} frames only! ")
        # nuc_mip_movie = nuc_mip_movie[:n_frames, :, :]
        # staygold_3d_frame_paths = staygold_3d_frame_paths[:n_frames]

        all_frame_patch_centers = []
        clean_mips = []

        for i in range(nuc_mip_movie.shape[0]):
            frame_patch_centers = []

            if i == 0:
                show_centroids = True
            elif i == nuc_mip_movie.shape[0]-1:
                show_centroids = True
            else:
                show_centroids = False

            raw_frame_patch_centers, clean_imgs, labels = patch_centers_from_nucleus(nuc_mip_movie[i, :, :],
                                                          show_centroids=show_centroids,
                                                          min_coord_dist=min_coord_dist,
                                                          local_threshold_block_size=local_threshold_block_size,
                                                          local_threshold_sigma=local_threshold_sigma,
                                                          gaussian_filter_sigma=gaussian_filter_sigma
                                                          )

            ### remove intensity outlier cells ###
            staygold_frame_3d = imread(staygold_3d_frame_paths[i])

            int_low = 0
            int_high = np.percentile(mito_mip_movie[i, :, :], 99)
            staygold_mip = np.clip(mito_mip_movie[i, :, :], int_low, int_high)

            staygold_normalized = (staygold_mip - int_low) / (int_high - int_low)
            staygold_normalized = np.clip(staygold_normalized, 0, 1)
            staygold_lcn = normalize_local_contrast(staygold_normalized, sigma=128)

            print("Filtering patch centers based on StayGold intensity...")
            for cell_id, center in enumerate(raw_frame_patch_centers):

                patch = staygold_frame_3d[:, center[0] - 128:center[0] + 128, center[1] - 128:center[1] + 128]

                # remove too dim/bright cells
                staygold_mean = np.mean(patch[patch>filters.threshold_otsu(patch)])
                staygold_mean_list.append(int(staygold_mean))

                frame_patch_centers.append(raw_frame_patch_centers[cell_id])
                # if staygold_mean < 20000 and staygold_mean > 5000:
                # if staygold_mean < 20000 and staygold_mean > 1000:
                #     frame_patch_centers.append(raw_frame_patch_centers[cell_id])
                # else:
                #     print("Outside range")

                ### plot selected cells ###
            filtered_patch_centers_2d = np.array(frame_patch_centers)

            cmap = plt.get_cmap('tab20')
            colors = []
            for i in range(labels.max()):
                colors.append(cmap(i%20))

            overlay = color.label2rgb(labels, image=staygold_lcn, colors=colors, bg_label=0, alpha=0.4)

            plt.figure(figsize=(20, 6))
            plt.imshow(overlay)
            plt.scatter(filtered_patch_centers_2d[:, 1], filtered_patch_centers_2d[:, 0], c='green', s=15)
            plt.axis('off')
            plt.show()

            all_frame_patch_centers.append(frame_patch_centers)
        all_region_patch_centers.append(all_frame_patch_centers)
    all_sample_region_patch_centers.append(all_region_patch_centers)

In [5]:
import pickle as pkl
# with open('andre_3color_celldeath_all_patch_centers.pkl', 'wb') as f:
#     pkl.dump(all_sample_region_patch_centers, f)

with open('andre_3color_celldeath_all_patch_centers.pkl', 'rb') as f:
    all_sample_region_patch_centers = pkl.load(f)

In [6]:
len(all_sample_region_patch_centers[0][0])

57

In [ ]:
# region1_patch_centers = all_sample_region_patch_centers[0][0]
# region2_patch_centers = all_sample_region_patch_centers[0][1]

In [ ]:
# reverse the time-order for region1 patch centers
# all_sample_region_patch_centers[0][0] = all_sample_region_patch_centers[0][0][::-1]
# all_sample_region_patch_centers[0][1] = all_sample_region_patch_centers[0][1][::-1]

In [ ]:
# # ORIGINAL WORKING COPY 11min/cell :(
# save_root = "/mnt/aquila/ssd_processing/Others/MitoSpace4D/andre_3color_cancer"
# metadata = pd.DataFrame(columns=['sample id', 'region id', 'cell id', 'movie id', 'cell center (x,y)'])
# offset = 0
# for sample_id, sample_region_patch_centers in enumerate(all_sample_region_patch_centers):
#
#     date = all_date[sample_id]
#
#     data_path = sorted(glob(root_path + 'Data2601-/MOSAIC_Data/Processed_Data/*'+date+'*Andre 3 color Cell Death/Sample '+all_sample_id[sample_id]+'/'))[0]
#
#     save_dir = osp.join(save_root, f"{date}-{sample_id}")
#     os.makedirs(save_dir, exist_ok=True)
#
#     print(os.path.exists(data_path))
#
#     all_region_paths = sorted(glob(data_path + '?'))
#     for region_id, region_patch_centers in enumerate(sample_region_patch_centers):
#
#         print(f"Processing region {region_id}...")
#         region_path = all_region_paths[region_id]
#         mito_paths = sorted(glob(region_path + '/*488nm*tif'))
#         mem_paths = sorted(glob(region_path + '/*560nm*tif'))
#         nuc_paths = sorted(glob(region_path + '/*642nm*tif'))
#
#         rev_region_patch_centers = region_patch_centers[::-1]
#
#         ids = [i for i in range(len(rev_region_patch_centers[0]))]
#         cell_centers = {i: [rev_region_patch_centers[0][i]] for i in ids}
#
#         completed_cell_ids = []
#         # iterate over the frames
#         for frame_id in range(1, len(rev_region_patch_centers)):
#             current_frame_centers = rev_region_patch_centers[frame_id][:]
#
#             # for each existing cell, find the closest center in the current frame
#             for cell_id in cell_centers.keys():
#
#                 if cell_id in completed_cell_ids:
#                     continue
#
#                 last_center = cell_centers[cell_id][-1]
#
#                 # compute distances to all centers in the current frame
#                 distances = [np.linalg.norm(np.array(last_center) - np.array(center)) for center in current_frame_centers]
#
#                 # find the index of the closest center
#                 if distances:
#                     min_index = np.argmin(distances)
#                     min_distance = distances[min_index]
#
#                     # set a threshold to avoid incorrect matches (e.g., max distance of 50 pixels)
#                     if min_distance < 50:
#                         matched_center = current_frame_centers[min_index]
#                         cell_centers[cell_id].append(matched_center)
#
#                         # remove the matched center from the list to prevent multiple assignments
#                         current_frame_centers.pop(min_index)
#                     else:
#                         # if no match found within threshold, add to completed list
#                         completed_cell_ids.append(cell_id)
#                 else:
#                     # if no centers left to match, mark cell as completed
#                     completed_cell_ids.append(cell_id)
#
#         # invert the cell centers back to original time order
#         cell_centers = {cell_id: centers[::-1] for cell_id, centers in cell_centers.items()}
#
#         # Iterate, extract and save
#         for cell_id, centers in cell_centers.items():
#             n_frames = len(centers)
#             print(n_frames)
#
#             start_frame = len(mito_paths) - n_frames # reverse order b/c of tracking
#
#             movie = np.empty((3, n_frames, 60, 256, 256), dtype=np.float32)
#
#             for i in range(n_frames):
#                 print(f"Frame {i}")
#                 # print(centers)
#                 metadata.loc[len(metadata)] = [sample_id, region_id, cell_id, mito_paths[start_frame+i].split('/')[-1].split('.')[0], centers[i]]
#
#                 # Extract the mito image from the region path
#                 mito_path = mito_paths[start_frame+i]
#                 mito_img = imread(mito_path)
#                 mito_patch, center_z = process_one_cell(centers[i], mito_img)
#
#                 # centers[i] = [center_z[0], centers[i][1], centers[i][0]]
#
#                 # Extract the nucleus image from the region path
#                 nuc_path = nuc_paths[start_frame+i]
#                 nuc_img = imread(nuc_path)
#                 # nuc_patch = crop_patch(nuc_img, centers[i])
#                 nuc_patch = crop_patch(nuc_img, center_z)
#
#                 # Extract the membrane image from the region path
#                 mem_path = mem_paths[start_frame+i]
#                 mem_img = imread(mem_path)
#                 # mem_patch = crop_patch(mem_img, centers[i])
#                 mem_patch = crop_patch(mem_img, center_z)
#
#                 movie[:, i, :, :, :] = np.stack([mito_patch, mem_patch, nuc_patch], axis=0)
#
#             # movie_path = os.path.join(save_dir, f"{str(cell_id+offset).zfill(6)}-0.npy")
#             movie_path = os.path.join(save_dir, f"{str(cell_id).zfill(6)}-0.npy")
#
#             np.save(movie_path, movie)
#             offset += 1
#             print(f"Saved movie for cell {cell_id}/{len(cell_centers.keys())}")
#         print(f"Saved {len(cell_centers)} cells for region {region_id}.")
#     # Metadata
#     metadata.to_csv(os.path.join(save_dir, 'metadata.csv'), index=False)
#     print(f"Saved metadata to {os.path.join(save_dir, 'metadata.csv')}")

In [ ]:
# # Version 2
# save_root = "/mnt/aquila/ssd_processing/Others/MitoSpace4D/andre_3color_cancer"
# metadata = pd.DataFrame(columns=['sample_id', 'region_id', 'cell_id', 'cell_center_zyx', 'frame_id', 'path'])
#
# cell_index_offset = 0
# for sample_id, sample_region_patch_centers in enumerate(all_sample_region_patch_centers):
#
#     date = all_date[sample_id]
#
#     data_path = sorted(glob(root_path + 'Data2601-/MOSAIC_Data/Processed_Data/*'+date+'*Andre 3 color Cell Death/Sample '+all_sample_id[sample_id]+'/'))[0]
#
#     save_dir = osp.join(save_root, f"{date}-{sample_id}")
#     os.makedirs(save_dir, exist_ok=True)
#
#     all_region_paths = sorted(glob(data_path + '?'))
#     for region_id, region_patch_centers in enumerate(sample_region_patch_centers):
#
#         print(f"Processing region {region_id}...")
#         region_path = all_region_paths[region_id]
#         mito_paths = sorted(glob(region_path + '/*488nm*tif'))
#         mem_paths = sorted(glob(region_path + '/*560nm*tif'))
#         nuc_paths = sorted(glob(region_path + '/*642nm*tif'))
#
#         rev_region_patch_centers = region_patch_centers[::-1]
#
#         ids = [i for i in range(len(rev_region_patch_centers[0]))]
#         cell_centers = {i: [rev_region_patch_centers[0][i]] for i in ids}
#
#         # completed_cell_ids = []
#         # iterate over the frames
#         for frame_id in range(1, len(rev_region_patch_centers)):
#             current_frame_centers = rev_region_patch_centers[frame_id][:]
#
#             # for each existing cell, find the closest center in the current frame
#             for cell_id in cell_centers.keys():
#
#                 last_center = cell_centers[cell_id][-1]
#
#                 if last_center is None:
#                     # Append None for this frame
#                     cell_centers[cell_id].append(None)
#                 else:
#                     # compute distances to all centers in the current frame
#                     distances = [np.linalg.norm(np.array(last_center) - np.array(center)) for center in current_frame_centers]
#
#                     # find the index of the closest center
#                     if distances:
#                         min_index = np.argmin(distances)
#                         min_distance = distances[min_index]
#
#                         # set a threshold to avoid incorrect matches (e.g., max distance of 50 pixels)
#                         if min_distance < 50:
#                             matched_center = current_frame_centers[min_index]
#                             cell_centers[cell_id].append(matched_center)
#
#                             # remove the matched center from the list to prevent multiple assignments
#                             current_frame_centers.pop(min_index)
#                         else:
#                             # if no match found within threshold, add to completed list
#                             # completed_cell_ids.append(cell_id)
#                             cell_centers[cell_id].append(None)
#                     else:
#                         # if no centers left to match, mark cell as completed
#                         # completed_cell_ids.append(cell_id)
#                         cell_centers[cell_id].append(None)
#
#         # invert the cell centers back to original time order
#         cell_centers = {cell_id: centers[::-1] for cell_id, centers in cell_centers.items()}
#
#         # Create directories for each cell
#         for cell_id, centers in cell_centers.items():
#             if region_id == 0:
#                 cell_dir = osp.join(save_dir, str(cell_id).zfill(6))
#             else:
#                 cell_dir = osp.join(save_dir, str(cell_id+cell_index_offset).zfill(6))
#             os.makedirs(cell_dir, exist_ok=True)
#
#         # Iterate, extract and save
#         n_frames = len(mito_paths)
#         for frame_id in trange(n_frames):
#
#             # Load images
#             mito_path = mito_paths[frame_id]
#             mito_img = imread(mito_path)
#
#             nuc_path = nuc_paths[frame_id]
#             nuc_img = imread(nuc_path)
#
#             mem_path = mem_paths[frame_id]
#             mem_img = imread(mem_path)
#
#             for cell_id, centers in tqdm(cell_centers.items()):
#
#                 cell_center = centers[int(frame_id)]
#                 if cell_center is None:
#                     continue
#
#                 mito_patch, center_zyx = process_one_cell(cell_center, mito_img)
#                 mem_patch = crop_patch(mem_img, center_zyx)
#                 nuc_patch = crop_patch(nuc_img, center_zyx)
#
#                 frame = np.stack([mito_patch, mem_patch, nuc_patch], axis=0)
#                 if region_id == 0:
#                     cell_dir = str(cell_id).zfill(6)
#                     cell_index_offset += 1
#                 else:
#                     cell_dir = str(cell_id + cell_index_offset).zfill(6)
#
#                 frame_id = str(frame_id).zfill(3)
#                 outfile = osp.join(save_dir, cell_dir, f"frame_{frame_id}.npy")
#                 np.save(outfile, frame)
#                 metadata.loc[len(metadata)] = [sample_id, region_id, cell_id, frame_id, center_zyx, osp.join(cell_dir, f"frame_{frame_id}.npy")]
#                 # print(f"Saved frame {frame_id} for cell {cell_id}")
#         print(f"Saved {len(cell_centers)} cells for region {region_id}.")
#         metadata.to_csv(osp.join(save_dir, 'metadata.csv'), index=False)
#
#     # Metadata
#     metadata.to_csv(os.path.join(save_dir, 'metadata.csv'), index=False)
#     print(f"Saved metadata to {os.path.join(save_dir, 'metadata.csv')}")

In [ ]:
# import os
# import os.path as osp
# import numpy as np
# import pandas as pd
# from glob import glob
# from tqdm import tqdm, trange
# from skimage.io import imread
# from concurrent.futures import ProcessPoolExecutor, as_completed
# import multiprocessing
#
# # --- Worker Function for Multiprocessing ---
# # This must be defined at the top level to be picklable by multiprocessing
# def process_frame_task(frame_idx, mito_path, nuc_path, mem_path, cell_centers_map, save_dir, region_offset, region_id, sample_id):
#     """
#     Processes a single frame: loads images, crops all cells active in this frame,
#     saves .npy files, and returns metadata records.
#     """
#     try:
#         # Load images (Heavy I/O)
#         mito_img = imread(mito_path)
#         # nuc_img = imread(nuc_path)
#         # mem_img = imread(mem_path)
#
#         frame_metadata = []
#
#         # Iterate over all cells tracked in this region
#         for local_cell_id, centers in cell_centers_map.items():
#
#             # Check if cell exists in this frame
#             if frame_idx >= len(centers):
#                 continue
#
#             cell_center = centers[int(frame_idx)]
#             if cell_center is None:
#                 continue
#
#             # --- ID Calculation ---
#             # Calculate Global Cell ID based on region offset
#             if region_id == 0:
#                 global_cell_id = local_cell_id
#             else:
#                 global_cell_id = local_cell_id + region_offset
#
#             # Create directory logic
#             cell_dir_name = str(global_cell_id).zfill(6)
#             cell_full_dir = osp.join(save_dir, cell_dir_name)
#             os.makedirs(cell_full_dir, exist_ok=True)
#
#             # --- Processing (CPU Bound) ---
#             # specific processing functions assumed to be defined globally or imported
#             mito_patch, center_zyx = process_one_cell(cell_center, mito_img)
#             # mem_patch = crop_patch(mem_img, center_zyx)
#             # nuc_patch = crop_patch(nuc_img, center_zyx)
#
#             # Stack and Save
#             # frame_data = np.stack([mito_patch, mem_patch, nuc_patch], axis=0)
#             frame_data = mito_patch
#
#             frame_str = str(frame_idx).zfill(3)
#             filename = f"frame_{frame_str}.npy"
#             outfile = osp.join(cell_full_dir, filename)
#
#             np.save(outfile, frame_data)
#
#             # Record Metadata
#             frame_metadata.append({
#                 'sample_id': sample_id,
#                 'region_id': region_id,
#                 'cell_id': global_cell_id,
#                 'frame_id': frame_str,
#                 'cell_center_zyx': center_zyx,
#                 'path': outfile
#             })
#
#         return frame_metadata
#
#     except Exception as e:
#         print(f"Error processing frame {frame_idx}: {e}")
#         return []
#
# # --- Main Execution ---
# if __name__ == '__main__':
#
#     # Configuration
#     save_root = "/mnt/aquila/ssd_processing/Others/MitoSpace4D/andre_3color_cancer"
#     # Use ~80% of available cores to prevent system freeze during heavy I/O
#     # num_workers = max(1, int(multiprocessing.cpu_count() * 0.8))
#     num_workers = 4
#
#     # Initialize global tracking
#     global_metadata_list = []
#     total_cells_processed_so_far = 0
#
#     for sample_id, sample_region_patch_centers in enumerate(all_sample_region_patch_centers):
#
#         date = all_date[sample_id]
#
#         # Find Data Path
#         # Note: Added safety check for empty glob results
#         search_pattern = root_path + 'Data2601-/MOSAIC_Data/Processed_Data/*' + date + '*Andre 3 color Cell Death/Sample ' + all_sample_id[sample_id] + '/'
#         found_paths = sorted(glob(search_pattern))
#         if not found_paths:
#             print(f"Warning: No data found for {search_pattern}")
#             continue
#         data_path = found_paths[0]
#
#         save_dir = osp.join(save_root, f"{date}-{sample_id}")
#         os.makedirs(save_dir, exist_ok=True)
#
#         all_region_paths = sorted(glob(data_path + '?'))
#
#         for region_id, region_patch_centers in enumerate(sample_region_patch_centers):
#             print(f"Processing Sample {sample_id} | Region {region_id}...")
#
#             region_path = all_region_paths[region_id]
#             mito_paths = sorted(glob(region_path + '/*488nm*tif'))
#             mem_paths = sorted(glob(region_path + '/*560nm*tif'))
#             nuc_paths = sorted(glob(region_path + '/*642nm*tif'))
#
#             # --- Tracking Logic (Sequential, fast enough) ---
#             rev_region_patch_centers = region_patch_centers[::-1]
#             ids = [i for i in range(len(rev_region_patch_centers[0]))]
#             cell_centers = {i: [rev_region_patch_centers[0][i]] for i in ids}
#
#             for frame_id in range(1, len(rev_region_patch_centers)):
#                 current_frame_centers = rev_region_patch_centers[frame_id][:]
#                 for cell_id in cell_centers.keys():
#                     last_center = cell_centers[cell_id][-1]
#                     if last_center is None:
#                         cell_centers[cell_id].append(None)
#                     else:
#                         distances = [np.linalg.norm(np.array(last_center) - np.array(center)) for center in current_frame_centers]
#                         if distances:
#                             min_index = np.argmin(distances)
#                             min_distance = distances[min_index]
#                             if min_distance < 50:
#                                 matched_center = current_frame_centers[min_index]
#                                 cell_centers[cell_id].append(matched_center)
#                                 current_frame_centers.pop(min_index)
#                             else:
#                                 cell_centers[cell_id].append(None)
#                         else:
#                             cell_centers[cell_id].append(None)
#
#             # Invert back to original time order
#             cell_centers = {cell_id: centers[::-1] for cell_id, centers in cell_centers.items()}
#
#             # --- Parallel Frame Processing ---
#             n_frames = len(mito_paths)
#
#             # We capture the current offset before processing this region
#             current_region_offset = total_cells_processed_so_far
#
#             print(f"Starting parallel processing for {n_frames} frames using {num_workers} workers...")
#
#             with ProcessPoolExecutor(max_workers=num_workers) as executor:
#                 # Prepare arguments for each frame
#                 futures = []
#                 for frame_idx in range(n_frames):
#                     futures.append(executor.submit(
#                         process_frame_task,
#                         frame_idx,
#                         mito_paths[frame_idx],
#                         nuc_paths[frame_idx],
#                         mem_paths[frame_idx],
#                         cell_centers, # Pass the whole dict
#                         save_dir,
#                         current_region_offset,
#                         region_id,
#                         sample_id
#                     ))
#
#                 # Collect results as they finish
#                 for future in tqdm(as_completed(futures), total=n_frames, desc=f"Region {region_id}"):
#                     result_metadata = future.result()
#                     global_metadata_list.extend(result_metadata)
#
#             # Update the global cell counter only after region is fully processed
#             # (In original logic, offset seemed to track total cells across regions)
#             total_cells_processed_so_far += len(cell_centers)
#
#             print(f"Finished Region {region_id}. Total unique cells processed so far: {total_cells_processed_so_far}")
#
#         # --- Save Metadata ---
#         # Create DataFrame once at the end of the sample (or could be moved to very end of script)
#         print("Compiling metadata...")
#         metadata = pd.DataFrame(global_metadata_list)
#         metadata_path = osp.join(save_dir, 'metadata.csv')
#         metadata.to_csv(metadata_path, index=False)
#         print(f"Saved metadata to {metadata_path}")

In [ ]:
# import os
# import os.path as osp
# import numpy as np
# import pandas as pd
# from glob import glob
# from tqdm import tqdm, trange
# from skimage.io import imread
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import multiprocessing
#
# # --- Worker Function for Multithreading ---
# def process_patch_task(mito_img, nuc_img, mem_img, cell_center, local_cell_id,
#                        frame_idx, save_dir, sample_id, region_id, region_offset):
#     """
#     Processes a single patch for a single cell.
#     Because this is used in a ThreadPool, 'mito_img' is read from shared memory
#     without being copied, saving huge amounts of RAM.
#     """
#     try:
#         # --- ID Calculation ---
#         if region_id == 0:
#             global_cell_id = local_cell_id
#         else:
#             global_cell_id = local_cell_id + region_offset
#
#         # Create directory logic
#         cell_dir_name = str(global_cell_id).zfill(6)
#         cell_full_dir = osp.join(save_dir, cell_dir_name)
#         # Note: os.makedirs is thread-safe in Python 3 exist_ok=True
#         os.makedirs(cell_full_dir, exist_ok=True)
#
#         # --- Processing ---
#         # Assuming process_one_cell and crop_patch are defined globally
#         mito_patch, center_zyx = process_one_cell(cell_center, mito_img)
#
#         # Uncomment if you need other channels
#         mem_patch = crop_patch(mem_img, center_zyx)
#         nuc_patch = crop_patch(nuc_img, center_zyx)
#
#         # Stack and Save
#         frame_data = np.stack([mito_patch, mem_patch, nuc_patch], axis=0)
#         # frame_data = mito_patch
#
#         frame_str = str(frame_idx).zfill(3)
#         filename = f"frame_{frame_str}.npy"
#         outfile = osp.join(cell_full_dir, filename)
#
#         np.save(outfile, frame_data)
#
#         # Return Metadata Record
#         return {
#             'sample_id': sample_id,
#             'region_id': region_id,
#             'cell_id': global_cell_id,
#             'frame_id': frame_str,
#             'cell_center_zyx': center_zyx,
#             'path': outfile
#         }
#
#     except Exception as e:
#         print(f"Error processing cell {local_cell_id} at frame {frame_idx}: {e}")
#         return None
#
#
# # Configuration
# save_root = "/mnt/aquila/ssd_processing/Others/MitoSpace4D/andre_3color_cancer_3channels"
# # Threading has lower overhead than multiprocessing, so you can often use more workers.
# # However, if process_one_cell is CPU heavy, Python's GIL might limit gains.
# # If it's I/O bound (saving files), more workers is better.
# num_workers = 2
#
# # Initialize global tracking
# global_metadata_list = []
# total_cells_processed_so_far = 0
#
# for sample_id, sample_region_patch_centers in enumerate(all_sample_region_patch_centers):
#
#     date = all_date[sample_id]
#
#     # Find Data Path
#     search_pattern = root_path + 'Data2601-/MOSAIC_Data/Processed_Data/*' + date + '*Andre 3 color Cell Death/Sample ' + all_sample_id[sample_id] + '/'
#     found_paths = sorted(glob(search_pattern))
#     if not found_paths:
#         print(f"Warning: No data found for {search_pattern}")
#         continue
#     data_path = found_paths[0]
#
#     save_dir = osp.join(save_root, f"{date}-{sample_id}")
#     os.makedirs(save_dir, exist_ok=True)
#
#     all_region_paths = sorted(glob(data_path + '?'))
#
#     for region_id, region_patch_centers in enumerate(sample_region_patch_centers):
#         print(f"Processing Sample {sample_id} | Region {region_id}...")
#
#         region_path = all_region_paths[region_id]
#         mito_paths = sorted(glob(region_path + '/*488nm*tif'))
#         mem_paths = sorted(glob(region_path + '/*560nm*tif'))
#         nuc_paths = sorted(glob(region_path + '/*642nm*tif'))
#
#         # --- Tracking Logic ---
#         rev_region_patch_centers = region_patch_centers[::-1]
#         ids = [i for i in range(len(rev_region_patch_centers[0]))]
#         cell_centers = {i: [rev_region_patch_centers[0][i]] for i in ids}
#
#         for frame_id in range(1, len(rev_region_patch_centers)):
#             current_frame_centers = rev_region_patch_centers[frame_id][:]
#             for cell_id in cell_centers.keys():
#                 last_center = cell_centers[cell_id][-1]
#                 if last_center is None:
#                     cell_centers[cell_id].append(None)
#                 else:
#                     distances = [np.linalg.norm(np.array(last_center) - np.array(center)) for center in current_frame_centers]
#                     if distances:
#                         min_index = np.argmin(distances)
#                         min_distance = distances[min_index]
#                         if min_distance < 50:
#                             matched_center = current_frame_centers[min_index]
#                             cell_centers[cell_id].append(matched_center)
#                             current_frame_centers.pop(min_index)
#                         else:
#                             cell_centers[cell_id].append(None)
#                     else:
#                         cell_centers[cell_id].append(None)
#
#         # Invert back to original time order
#         cell_centers = {cell_id: centers[::-1] for cell_id, centers in cell_centers.items()}
#
#         # --- Sequential Frame Loading / Parallel Patch Processing ---
#         n_frames = len(mito_paths)
#         current_region_offset = total_cells_processed_so_far
#
#         print(f"Processing {n_frames} frames. Image load is sequential, patch extraction is threaded ({num_workers} workers).")
#
#         # We use a ThreadPoolExecutor.
#         # Note: We create/shutdown the executor per region or use a context manager to ensure cleanup.
#         # Using it per frame might have slight overhead, but ensures we wait for all patches
#         # to finish before dropping the image memory.
#
#         with ThreadPoolExecutor(max_workers=num_workers) as executor:
#             for frame_idx in trange(n_frames, desc=f"Reg {region_id} Frames"):
#
#                 # 1. Load Images (Main Thread - Single Copy in RAM)
#                 try:
#                     mito_img = imread(mito_paths[frame_idx])
#                     nuc_img = imread(nuc_paths[frame_idx])
#                     mem_img = imread(mem_paths[frame_idx])
#                     # nuc_img = None # Placeholders if keeping commented out
#                     # mem_img = None
#                 except IndexError:
#                     print(f"Frame {frame_idx} missing from paths.")
#                     continue
#
#                 # 2. Submit Patch Tasks
#                 futures = []
#                 for local_cell_id, centers in cell_centers.items():
#
#                     # Validate frame existence for this cell
#                     if frame_idx >= len(centers):
#                         continue
#
#                     cell_center = centers[int(frame_idx)]
#                     if cell_center is None:
#                         continue
#
#                     # Submit task
#                     futures.append(executor.submit(
#                         process_patch_task,
#                         mito_img, # Passed by reference (cheap in threads)
#                         nuc_img,
#                         mem_img,
#                         cell_center,
#                         local_cell_id,
#                         frame_idx,
#                         save_dir,
#                         sample_id,
#                         region_id,
#                         current_region_offset
#                     ))
#
#                 # 3. Collect Results for this Frame
#                 # We wait for all patches in this frame to finish before loading the next frame
#                 # This prevents RAM usage from spiraling out of control.
#                 for future in as_completed(futures):
#                     res = future.result()
#                     if res:
#                         global_metadata_list.append(res)
#
#         # Update global counter after region finishes
#         total_cells_processed_so_far += len(cell_centers)
#         print(f"Finished Region {region_id}. Total unique cells: {total_cells_processed_so_far}")
#
#     # --- Save Metadata ---
#     print("Compiling metadata...")
#     metadata = pd.DataFrame(global_metadata_list)
#     metadata_path = osp.join(save_dir, 'metadata.csv')
#     metadata.to_csv(metadata_path, index=False)
#     print(f"Saved metadata to {metadata_path}")

In [7]:
import os
import os.path as osp
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm, trange
from skimage.io import imread
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing
import gc

# --- Worker Function for Multithreading ---
def process_patch_task(mito_img, nuc_img, mem_img, cell_center, local_cell_id,
                       frame_idx, save_dir, sample_id, region_id, region_offset):
    """
    Processes a single patch for a single cell.
    Because this is used in a ThreadPool, 'mito_img' is read from shared memory
    without being copied, saving huge amounts of RAM.
    """
    try:
        # --- ID Calculation ---
        if region_id == 0:
            global_cell_id = local_cell_id
        else:
            global_cell_id = local_cell_id + region_offset

        # Create directory logic
        cell_dir_name = str(global_cell_id).zfill(6)
        cell_full_dir = osp.join(save_dir, cell_dir_name)
        # Note: os.makedirs is thread-safe in Python 3 exist_ok=True
        os.makedirs(cell_full_dir, exist_ok=True)

        # --- Processing ---
        # Assuming process_one_cell and crop_patch are defined globally in a previous cell
        mito_patch, center_zyx = process_one_cell(cell_center, mito_img)

        # Uncomment if you need other channels
        mem_patch = crop_patch(mem_img, center_zyx)
        nuc_patch = crop_patch(nuc_img, center_zyx)

        # Stack and Save
        frame_data = np.stack([mito_patch, mem_patch, nuc_patch], axis=0)
        # frame_data = mito_patch

        frame_str = str(frame_idx).zfill(3)
        filename = f"frame_{frame_str}.npy"
        outfile = osp.join(cell_full_dir, filename)

        np.save(outfile, frame_data)

        # Return Metadata Record
        return {
            'sample_id': sample_id,
            'region_id': region_id,
            'cell_id': global_cell_id,
            'frame_id': frame_str,
            'cell_center_zyx': center_zyx,
            'path': outfile
        }

    except Exception as e:
        print(f"Error processing cell {local_cell_id} at frame {frame_idx}: {e}")
        return None


if __name__ == '__main__':
    # Configuration
    save_root = "/mnt/aquila/ssd_processing/Others/MitoSpace4D/andre_3color_cancer_3channels"

    # In Jupyter, Threading is much safer than Multiprocessing because functions defined
    # in cells are not easily pickleable for separate processes.
    num_workers = 1

    # Initialize global tracking
    global_metadata_list = []
    total_cells_processed_so_far = 0

    # NOTE: Ensure 'all_sample_region_patch_centers', 'all_date', 'root_path', 'all_sample_id'
    # are defined in a previous cell.
    for sample_id, sample_region_patch_centers in enumerate(all_sample_region_patch_centers):

        date = all_date[sample_id]

        # Find Data Path
        search_pattern = root_path + 'Data2601-/MOSAIC_Data/Processed_Data/*' + date + '*Andre 3 color Cell Death/Sample ' + all_sample_id[sample_id] + '/'
        found_paths = sorted(glob(search_pattern))
        if not found_paths:
            print(f"Warning: No data found for {search_pattern}")
            continue
        data_path = found_paths[0]

        save_dir = osp.join(save_root, f"{date}-{sample_id}")
        os.makedirs(save_dir, exist_ok=True)

        all_region_paths = sorted(glob(data_path + '?'))

        for region_id, region_patch_centers in enumerate(sample_region_patch_centers):
            print(f"Processing Sample {sample_id} | Region {region_id}...")

            region_path = all_region_paths[region_id]
            mito_paths = sorted(glob(region_path + '/*488nm*tif'))
            mem_paths = sorted(glob(region_path + '/*560nm*tif'))
            nuc_paths = sorted(glob(region_path + '/*642nm*tif'))

            # --- Tracking Logic ---
            rev_region_patch_centers = region_patch_centers[::-1]
            ids = [i for i in range(len(rev_region_patch_centers[0]))]
            cell_centers = {i: [rev_region_patch_centers[0][i]] for i in ids}

            for frame_id in range(1, len(rev_region_patch_centers)):
                current_frame_centers = rev_region_patch_centers[frame_id][:]
                for cell_id in cell_centers.keys():
                    last_center = cell_centers[cell_id][-1]
                    if last_center is None:
                        cell_centers[cell_id].append(None)
                    else:
                        distances = [np.linalg.norm(np.array(last_center) - np.array(center)) for center in current_frame_centers]
                        if distances:
                            min_index = np.argmin(distances)
                            min_distance = distances[min_index]
                            if min_distance < 50:
                                matched_center = current_frame_centers[min_index]
                                cell_centers[cell_id].append(matched_center)
                                current_frame_centers.pop(min_index)
                            else:
                                cell_centers[cell_id].append(None)
                        else:
                            cell_centers[cell_id].append(None)

            # Invert back to original time order
            cell_centers = {cell_id: centers[::-1] for cell_id, centers in cell_centers.items()}

            # --- Sequential Frame Loading / Parallel Patch Processing ---
            n_frames = len(mito_paths)
            current_region_offset = total_cells_processed_so_far

            print(f"Processing {n_frames} frames. Image load is sequential, patch extraction is threaded ({num_workers} workers).")

            # ThreadPoolExecutor cleans up threads automatically when exiting the 'with' block
            with ThreadPoolExecutor(max_workers=num_workers) as executor:
                for frame_idx in trange(n_frames, desc=f"Reg {region_id} Frames"):

                    # 1. Load Images (Main Thread - Single Copy in RAM)
                    try:
                        mito_img = imread(mito_paths[frame_idx])
                        nuc_img = imread(nuc_paths[frame_idx])
                        mem_img = imread(mem_paths[frame_idx])
                    except IndexError:
                        print(f"Frame {frame_idx} missing from paths.")
                        continue

                    # 2. Submit Patch Tasks
                    futures = []
                    for local_cell_id, centers in cell_centers.items():

                        # Validate frame existence for this cell
                        if frame_idx >= len(centers):
                            continue

                        cell_center = centers[int(frame_idx)]
                        if cell_center is None:
                            continue

                        # Submit task
                        futures.append(executor.submit(
                            process_patch_task,
                            mito_img,
                            nuc_img,
                            mem_img,
                            cell_center,
                            local_cell_id,
                            frame_idx,
                            save_dir,
                            sample_id,
                            region_id,
                            current_region_offset
                        ))

                    # 3. Collect Results for this Frame
                    for future in as_completed(futures):
                        res = future.result()
                        if res:
                            global_metadata_list.append(res)

            # --- Explicit Cleanup ---
            # Ensure workers are deleted and memory is freed immediately after the region is done
            del executor
            gc.collect()

            # Update global counter after region finishes
            total_cells_processed_so_far += len(cell_centers)
            print(f"Finished Region {region_id}. Total unique cells: {total_cells_processed_so_far}")

    # --- Save Metadata ---
    print("Compiling metadata...")
    metadata = pd.DataFrame(global_metadata_list)
    metadata_path = osp.join(save_dir, 'metadata.csv')
    metadata.to_csv(metadata_path, index=False)
    print(f"Saved metadata to {metadata_path}")

Processing Sample 0 | Region 0...
Processing 57 frames. Image load is sequential, patch extraction is threaded (1 workers).


Reg 0 Frames:  14%|█▍        | 8/57 [15:02:32<92:08:03, 6769.05s/it]


KeyboardInterrupt: 

In [ ]:
len(metadata)

In [ ]:
metadata.to_csv(os.path.join(save_dir, 'metadata.csv'), index=False)

Dev Below

In [ ]:
raw_coords = []
for frame_id in range(len(rev_region1_patch_centers)):
    current_frame_centers = rev_region1_patch_centers[frame_id]
    for center in current_frame_centers:
        raw_coords.append([center[1], center[0], frame_id])

raw_coords = np.array(raw_coords)

In [ ]:
ids = [i for i in range(len(rev_region1_patch_centers[0]))]
cell_centers = {i: [rev_region1_patch_centers[0][i]] for i in ids}
cell_centers

In [ ]:
# cell_centers = {i: [rev_region1_patch_centers[i]] for i in range(len(rev_region1_patch_centers[0]))}
completed_cell_ids = []
# iterate over the frames
for frame_id in range(1, len(rev_region1_patch_centers)):
    current_frame_centers = rev_region1_patch_centers[frame_id]

    # for each existing cell, find the closest center in the current frame
    for cell_id in cell_centers.keys():

        if cell_id in completed_cell_ids:
            continue

        last_center = cell_centers[cell_id][-1]

        # compute distances to all centers in the current frame
        distances = [np.linalg.norm(np.array(last_center) - np.array(center)) for center in current_frame_centers]

        # find the index of the closest center
        if distances:
            min_index = np.argmin(distances)
            min_distance = distances[min_index]

            # set a threshold to avoid incorrect matches (e.g., max distance of 50 pixels)
            if min_distance < 50:
                matched_center = current_frame_centers[min_index]
                cell_centers[cell_id].append(matched_center)

                # remove the matched center from the list to prevent multiple assignments
                current_frame_centers.pop(min_index)
            else:
                # if no match found within threshold, add to completed list
                completed_cell_ids.append(cell_id)
        else:
            # if no centers left to match, mark cell as completed
            completed_cell_ids.append(cell_id)

In [ ]:
import open3d as o3d

pcds = []
all_coords = []
for cell_id, centers in cell_centers.items():
    coords = []
    for frame_id, center in enumerate(centers):
        coords.append([center[1], center[0], frame_id])

    coords = np.array(coords)
    all_coords.append(coords)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(coords)
    # paint a random color for visualization
    pcd.paint_uniform_color([np.random.rand(), np.random.rand(), np.random.rand()])
    pcds.append(pcd)
all_coords = all_coords
o3d.visualization.draw_geometries(pcds)

# Main Processing Loop (WIP)